In [42]:
import os 
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")
# gemini_api_key

In [29]:
# reading data/documents

file_path = "swami_vivekananda.txt"
with open(file_path , 'r', encoding = "utf-8") as file:
    content = file.read()
    
    # print(content)

In [48]:
# chunking

# 1. Recursive method
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunked_content = splitter.create_documents([content])
# print(chunked_content)
# for i, ch in enumerate(chunked_content):
#     print(f"chunk {i} : {ch}\n\n")

In [31]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [32]:
from langchain_chroma import Chroma


vectorstoredb = Chroma.from_texts(texts=chunked_content, embedding=embeddings)
retriever = vectorstoredb.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [33]:
respond = retriever.invoke("What are the teachings of Swami Vivekananda?")

In [34]:
respond

[Document(id='373241cc-516a-48ae-ab3b-fab1f05cca7e', metadata={}, page_content='in themselves. For this they needed a life-giving, inspiring message. Swamiji found this message in the principle of the Atman, the doctrine of the potential divinity of the soul, taught in Vedanta, the ancient system of religious philosophy of India. He saw that, in spite of poverty, the masses clung to religion, but they had never been taught the life-giving, ennobling principles of Vedanta and how to apply them in practical life.'),
 Document(id='19f3fb2a-ac6f-45d2-aef2-96db773ff682', metadata={}, page_content='in themselves. For this they needed a life-giving, inspiring message. Swamiji found this message in the principle of the Atman, the doctrine of the potential divinity of the soul, taught in Vedanta, the ancient system of religious philosophy of India. He saw that, in spite of poverty, the masses clung to religion, but they had never been taught the life-giving, ennobling principles of Vedanta and 

In [35]:
respond[0].page_content


'in themselves. For this they needed a life-giving, inspiring message. Swamiji found this message in the principle of the Atman, the doctrine of the potential divinity of the soul, taught in Vedanta, the ancient system of religious philosophy of India. He saw that, in spite of poverty, the masses clung to religion, but they had never been taught the life-giving, ennobling principles of Vedanta and how to apply them in practical life.'

In [36]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise accordingly the dictionary below given.

Question: {question}
Context: {context}
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

In [37]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise accordingly the dictionary below given.\n\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [38]:
from langchain_google_genai import GoogleGenerativeAI
llm = GoogleGenerativeAI(model="models/gemini-2.5-flash")

In [39]:
llm.invoke("hi, i am manas, how are you?")

"Hi Manas! I'm doing great, thank you for asking.\n\nHow can I help you today?"

In [40]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

rag_chain = (
    {"context":retriever , "question":RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [41]:
res = rag_chain.invoke("Who is Swami Vivekananda?")

In [27]:
import textwrap

# text = """Swami Vivekananda's teachings offer a new understanding of religion..."""

print(textwrap.fill(res, width=80))

Swami Vivekananda, known in his pre-monastic life as Narendra Nath Datta, was
born in Kolkata on January 12, 1863. He hailed from an affluent family and was a
precocious boy, excelling in music, gymnastics, and studies, acquiring vast
knowledge. Born with a yogic temperament, he practiced meditation from boyhood
and was a disciple of Sri Ramakrishna. In 1890, he embarked on a long journey of
exploration and discovery across India.  He decided to attend the World’s
Parliament of Religions in Chicago in 1893, where his speeches made him famous
as an ‘orator by divine right’ and a ‘Messenger of Indian wisdom to the Western
world’. After the Parliament, he spent nearly three and a half years spreading
Vedanta in the USA and London. He returned to India in 1897, delivering lectures
to rouse religious consciousness and unify Hinduism. Swami Vivekananda made a
second visit to the West in 1899, spending time on the West coast of USA. He
returned to Belur Math in December 1900 and passed away o